# Step 2: summaries on Kaggle
Same files as the project, plus Ollama. Settings: GPU T4 x2, Internet on.

In [ ]:
!pip install -q openai tqdm
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time

env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="8192")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull llama3.1:8b

In [ ]:
%%writefile step1_preprocess.py
import gzip
import json
import urllib.request
from pathlib import Path

import pandas as pd

URL = "https://github.com/asappresearch/abcd/raw/master/data/abcd_v1.1.json.gz"
RAW_FILE = "data/abcd_v1.1.json.gz"


def get_subflow(flow, subflow):
    # the data splits the FAQ subflows into questions (boots_how_3 -> boots), giving 96 instead of 55
    if flow in ["single_item_query", "storewide_query"]:
        return subflow.split("_")[0]
    # only 3 conversations, not a subflow in the ABCD guidelines
    if subflow == "status_delivery_date":
        return "status_delivery_time"
    return subflow


Path("data").mkdir(exist_ok=True)
if not Path(RAW_FILE).exists():
    urllib.request.urlretrieve(URL, RAW_FILE)

with gzip.open(RAW_FILE, "rt", encoding="utf-8") as f:
    raw = json.load(f)

rows = []
for split in ["train", "dev", "test"]:
    for convo in raw[split]:
        flow = convo["scenario"]["flow"]
        subflow = convo["scenario"]["subflow"]

        # action turns are system messages that reveal the label, so they are left out
        full = [f"{speaker}: {text}" for speaker, text in convo["original"] if speaker != "action"]

        rows.append({
            "split": split,
            "convo_id": convo["convo_id"],
            "flow": flow,
            "subflow": get_subflow(flow, subflow),
            "full": "\n".join(full),
        })

data = pd.DataFrame(rows)
data.to_csv("data/conversations.csv", index=False)

# same data, but easy to read
with open("data/conversations.txt", "w", encoding="utf-8") as f:
    for _, row in data.sort_values(["flow", "subflow"]).iterrows():
        f.write(f"=== {row['convo_id']} | {row['split']} | {row['flow']} / {row['subflow']} ===\n")
        f.write(row["full"] + "\n\n")

print(data["split"].value_counts())
print("flows:", data["flow"].nunique(), "subflows:", data["subflow"].nunique())


In [ ]:
%%writefile step2_summarize.py
# python step2_summarize.py --split all --max-hours 8.5
import argparse
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI
from tqdm import tqdm

MODEL = "llama3.1:8b"
SUMMARY_FILE = "data/summaries.csv"
PROMPT = "Summarize this customer service conversation in 2-3 sentences. Call the customer \"the customer\", not by name. Write only the summary.\n\n{conversation}"

parser = argparse.ArgumentParser()
parser.add_argument("--split", default="dev", choices=["train", "dev", "test", "all"])
parser.add_argument("--max-hours", type=float, default=8.5)
args = parser.parse_args()

data = pd.read_csv("data/conversations.csv", keep_default_na=False)
if args.split == "all":
    data = data.sort_values("split")  # dev, test, then train
else:
    data = data[data["split"] == args.split]

# summaries are saved one by one, so a stopped run continues where it was
done = set()
if Path(SUMMARY_FILE).exists():
    done = set(pd.read_csv(SUMMARY_FILE)["convo_id"])

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
start = time.time()

for _, row in tqdm(data.iterrows(), total=len(data)):
    if row["convo_id"] in done:
        continue
    # stop before Kaggle ends the session, so the output is saved
    if time.time() - start > args.max_hours * 3600:
        print("stopped at the time limit, run again to continue")
        break
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": PROMPT.format(conversation=row["full"])}],
        temperature=0.0,
        max_tokens=200,
    )
    saved = {
        "convo_id": row["convo_id"],
        "summary": response.choices[0].message.content,
        "prompt_tokens": response.usage.prompt_tokens,
        "answer_tokens": response.usage.completion_tokens,
    }
    pd.DataFrame([saved]).to_csv(SUMMARY_FILE, mode="a", header=not Path(SUMMARY_FILE).exists(), index=False)


In [ ]:
!python step1_preprocess.py

In [ ]:
# continue from an earlier run: add its output with Add Input
!find /kaggle/input -name summaries.csv -exec cp {} data/ \;
!ls -la data

In [ ]:
!python step2_summarize.py --split all --max-hours 8.5